In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from data_setup import create_dataloaders

In [2]:
# Setup device agnostic
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

### 1. Load the data using Custom Dataset and DataLoader

In [3]:
csv_path = "data/processed_labels.csv"
data_dir = "data/lightcurves"

BATCH_SIZE = 32

In [4]:
import pandas as pd
df_check = pd.read_csv("data/processed_labels.csv")
print(df_check.columns)

Index(['TIC_ID', 'final_label'], dtype='object')


In [5]:
train_dataloader, test_dataloader = create_dataloaders(
    csv_path = csv_path,
    data_dir = data_dir,
    batch_size = BATCH_SIZE
)

train_dataloader, test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x118af9c10>,
 <torch.utils.data.dataloader.DataLoader at 0x1191f7b90>)

In [6]:
print(f"Number of training batches: {len(train_dataloader)}")
print(f"Number of testing batches: {len(test_dataloader)}")

Number of training batches: 57
Number of testing batches: 15


In [7]:
# Get a feature and label from the custom DataLoader
train_features_batch, train_labels_batch = next(iter(train_dataloader))
train_features_batch.shape, train_labels_batch.shape

(torch.Size([32, 1, 20000]), torch.Size([32]))

### 2. Model 0

In [8]:
class TESSModelV0(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.linear = nn.Linear(in_features = 20000, out_features = 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.linear(self.flatten(x)))

In [9]:
model_0 = TESSModelV0().to(device)
model_0

TESSModelV0(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear): Linear(in_features=20000, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [10]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model_0.parameters(), lr = 0.01)

In [11]:
def accuracy_fn(y_true, y_pred):
    y_pred_rounded = torch.round(y_pred)
    
    y_true_int = y_true.cpu().int()
    y_pred_int = y_pred_rounded.cpu().int()
    
    correct = torch.eq(y_true_int, y_pred_int).sum().item() 
    
    acc = (correct / len(y_pred)) * 100 
    return acc

In [12]:
epochs = 5

for epoch in range(epochs):
    train_loss, train_accuracy = 0, 0

    for batch, (X, y) in enumerate(train_dataloader):
        X, y = X.to(device), y.to(device)

        # Since BCE loss expects shape [32, 1] but y has shape [32], unsqueeze operation is required
        y = y.unsqueeze(dim = 1)

        y_pred = model_0(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss

        accuracy = accuracy_fn(y_true = y, y_pred = y_pred)
        train_accuracy += accuracy

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    train_loss /= len(train_dataloader)
    train_accuracy /= len(train_dataloader)

    print(f"Epoch: {epoch} | Train loss: {train_loss:.5f} | Train accuracy: {train_accuracy:.2f}%")
        

Epoch: 0 | Train loss: 20.43106 | Train accuracy: 47.73%
Epoch: 1 | Train loss: 20.71589 | Train accuracy: 47.93%
Epoch: 2 | Train loss: 20.69376 | Train accuracy: 47.94%
Epoch: 3 | Train loss: 20.67188 | Train accuracy: 47.95%
Epoch: 4 | Train loss: 20.64516 | Train accuracy: 47.98%


### 3. Model 1

In [13]:
class TESSModelV1(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()

        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features = input_shape, out_features = hidden_units),
            nn.ReLU(),
            nn.Linear(in_features = hidden_units, out_features = output_shape),
            nn.ReLU()
        )

    def forward(self, x):
        return self.layer_stack(x)

In [14]:
model_1 = TESSModelV1(input_shape = 20000, hidden_units = 10, output_shape = 1).to(device)
model_1

TESSModelV1(
  (layer_stack): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=20000, out_features=10, bias=True)
    (2): ReLU()
    (3): Linear(in_features=10, out_features=1, bias=True)
    (4): ReLU()
  )
)

In [15]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model_1.parameters(), lr = 0.005)

In [16]:
epochs = 10

for epoch in range(epochs):
    train_loss, train_accuracy = 0, 0

    for batch, (X, y) in enumerate(train_dataloader):
        X, y = X.to(device), y.to(device)

        # Since BCE loss expects shape [32, 1] but y has shape [32], unsqueeze operation is required
        y = y.unsqueeze(dim = 1)

        y_pred = model_1(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss

        accuracy = accuracy_fn(y_true = y, y_pred = y_pred)
        train_accuracy += accuracy

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    train_loss /= len(train_dataloader)
    train_accuracy /= len(train_dataloader)

    print(f"Epoch: {epoch} | Train loss: {train_loss:.5f} | Train accuracy: {train_accuracy:.2f}%")
        

Epoch: 0 | Train loss: 51.37972 | Train accuracy: 47.03%
Epoch: 1 | Train loss: 52.02473 | Train accuracy: 47.98%
Epoch: 2 | Train loss: 52.04741 | Train accuracy: 47.95%
Epoch: 3 | Train loss: 52.05876 | Train accuracy: 47.94%
Epoch: 4 | Train loss: 52.07577 | Train accuracy: 47.92%
Epoch: 5 | Train loss: 52.04741 | Train accuracy: 47.95%
Epoch: 6 | Train loss: 52.06443 | Train accuracy: 47.94%
Epoch: 7 | Train loss: 52.05309 | Train accuracy: 47.95%
Epoch: 8 | Train loss: 52.07577 | Train accuracy: 47.92%
Epoch: 9 | Train loss: 52.08144 | Train accuracy: 47.92%


### 4. 1D CNN Model

In [21]:
class TESSModelV2(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_block = nn.Sequential(
            nn.Conv1d(
                in_channels = 1,
                out_channels = 8,
                kernel_size = 16,
                stride = 2
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size = 4)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features = 8 * 2498, out_features = 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.classifier(self.conv_block(x))

model_2 = TESSModelV2().to(device)
model_2

TESSModelV2(
  (conv_block): Sequential(
    (0): Conv1d(1, 8, kernel_size=(16,), stride=(2,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=19984, out_features=1, bias=True)
    (2): Sigmoid()
  )
)

In [22]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model_2.parameters(), lr = 0.00005)

In [23]:
epochs = 30

for epoch in range(epochs):
    train_loss, train_accuracy = 0, 0

    for batch, (X, y) in enumerate(train_dataloader):
        X, y = X.to(device), y.to(device)

        # Since BCE loss expects shape [32, 1] but y has shape [32], unsqueeze operation is required
        y = y.unsqueeze(dim = 1)

        mean = X.mean(dim = 2, keepdim = True)
        std = X.std(dim = 2, keepdim = True)
        X = (X - mean) / (std + 1e-8) # 1e-8 prevents division by zero errors

        y_pred = model_2(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss

        accuracy = accuracy_fn(y_true = y, y_pred = y_pred)
        train_accuracy += accuracy

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    train_loss /= len(train_dataloader)
    train_accuracy /= len(train_dataloader)

    print(f"Epoch: {epoch} | Train loss: {train_loss:.5f} | Train accuracy: {train_accuracy:.2f}%")
        

Epoch: 0 | Train loss: 0.70525 | Train accuracy: 51.06%
Epoch: 1 | Train loss: 0.69016 | Train accuracy: 54.18%
Epoch: 2 | Train loss: 0.69817 | Train accuracy: 54.83%
Epoch: 3 | Train loss: 0.68348 | Train accuracy: 55.78%
Epoch: 4 | Train loss: 0.68144 | Train accuracy: 56.65%
Epoch: 5 | Train loss: 0.67899 | Train accuracy: 57.26%
Epoch: 6 | Train loss: 0.67539 | Train accuracy: 59.60%
Epoch: 7 | Train loss: 0.67984 | Train accuracy: 57.78%
Epoch: 8 | Train loss: 0.68260 | Train accuracy: 57.26%
Epoch: 9 | Train loss: 0.68142 | Train accuracy: 56.64%
Epoch: 10 | Train loss: 0.67210 | Train accuracy: 58.99%
Epoch: 11 | Train loss: 0.67464 | Train accuracy: 57.71%
Epoch: 12 | Train loss: 0.67411 | Train accuracy: 58.46%
Epoch: 13 | Train loss: 0.67078 | Train accuracy: 59.03%
Epoch: 14 | Train loss: 0.67455 | Train accuracy: 58.47%
Epoch: 15 | Train loss: 0.67573 | Train accuracy: 57.61%
Epoch: 16 | Train loss: 0.67325 | Train accuracy: 57.56%
Epoch: 17 | Train loss: 0.67225 | Train a

In [24]:
test_loss, test_accuracy = 0, 0

model_2.eval()
with torch.inference_mode():
    for X, y in test_dataloader:
        X, y = X.to(device), y.to(device)

        y = y.unsqueeze(dim = 1)

        mean = X.mean(dim = 2, keepdim = True)
        std = X.std(dim = 2, keepdim = True)
        X = (X - mean) / (std + 1e-8)

        test_pred = model_2(X)

        loss = loss_fn(test_pred, y)
        test_loss += loss

        accuracy = accuracy_fn(y_true = y, y_pred = test_pred)
        test_accuracy += accuracy

    test_loss /= len(test_dataloader)
    test_accuracy /= len(test_dataloader)

print(f"Final Test loss: {test_loss:.5f} | Final Test accuracy: {test_accuracy:.2f}%")

Final Test loss: 0.71520 | Final Test accuracy: 52.71%
